# DB Applications: Integration von relationalen CSV-Daten und RDF (SPARQL)
**Thema:** MLB Saison 2025 – Leistungsdaten im Kontext von Alter und Schlaghand

In diesem Notebook implementieren wir eine robuste Integrations-Architektur für heterogene Datenquellen:
1. **Lokale Datenbasis (CSV):** Wir lesen aktuelle Leistungsdaten der MLB-Saison 2025 ein (Home Runs, Batting Average).
2. **Globales Wissen (RDF/SPARQL):** Wir extrahieren die Namen der Top-Spieler und fragen über DBpedia deren Geburtsdatum sowie die präferierte Schlaghand (`dbp:bats`) ab.
3. **Relationale Integration (SQLite DB):** Beide Datenbestände werden als separate Tabellen in eine persistente SQLite-Datenbank (`baseball_2025.db`) geladen. Die semantische Zusammenführung (JOIN) und Auswertung erfolgt anschließend flexibel via SQL.

*Besonderheit im Data Cleaning:* Die rohen CSV-Daten weisen zunächst ein ungültiges Namensformat auf (Nachname, Vorname). Dies wird im ersten Schritt per Python bereinigt, um Kompatibilität mit dem Semantic Web (DBpedia) herzustellen.


In [27]:
# Benötigte Bibliotheken importieren
import pandas as pd
import sqlite3
from SPARQLWrapper import SPARQLWrapper, JSON

# 1. Lokale CSV-Datei 'stats.csv' einlesen
df_csv = pd.read_csv('stats.csv')

# --- DATENVORVERARBEITUNG (Data Cleaning) ---

# 1.1 Namen umdrehen (Aus "Judge, Aaron" wird "Aaron Judge")
# Wir teilen die Spalte 'last_name, first_name' am Komma auf.
# expand=True macht daraus zwei neue Spalten im Hintergrund (0=Nachname, 1=Vorname).
split_names = df_csv['last_name, first_name'].str.split(',', expand=True)

# Wir bauen die neue Spalte 'Name' zusammen: Vorname (Index 1) + Leerzeichen + Nachname (Index 0).
# .str.strip() entfernt störende Leerzeichen.
df_csv['Name'] = split_names[1].str.strip() + " " + split_names[0].str.strip()

# 1.2 Wichtige Spalten umbenennen, damit der SQL-Code später funktioniert
df_csv = df_csv.rename(columns={
    'home_run': 'HR',
    'batting_avg': 'AVG',
    'player_age': 'Age_DB' 
})

# Wir behalten nur die Spalten, die wir für unsere Applikation wirklich brauchen
# Das macht die Datenbank sauber und übersichtlich.
df_csv = df_csv[['Name', 'HR', 'AVG', 'year']]

print(f"Erfolg: 'stats.csv' wurde geladen und bereinigt. Datensatz enthält {len(df_csv)} Spieler.")

# Zur Kontrolle zeigen wir die sauberen Daten an
display(df_csv.head())

Erfolg: 'stats.csv' wurde geladen und bereinigt. Datensatz enthält 145 Spieler.


,Name,HR,AVG,year
0,Christian Walker,27,0.238,2025
1,Willson Contreras,20,0.257,2025
2,Jordan Beck,16,0.258,2025
3,Chase Meidroth,5,0.253,2025
4,Trevor Larnach,17,0.250,2025


## 1. Datenbeschaffung aus dem Semantic Web (RDF / SPARQL)

Wir filtern unsere Daten auf die Top-10-Spieler (Power Hitter) und bauen dynamisch einen SPARQL-Query für den DBpedia-Endpoint auf. Wir fragen zwei spezifische Attribute ab:
* `dbo:birthDate` (Ontology: Geburtsdatum)
* `dbp:bats` (Property: Schlaghand - Links, Rechts oder Beidhändig)

**Umgang mit Semantic Web Anomalien:** Um den bekannten "Virtuoso-Bug" (Fehlerhafte Aggregation bei leeren Feldern in DBpedia) zu umgehen, verzichten wir in SPARQL auf `MAX()` und `GROUP BY`. Wir laden stattdessen die rohen, ungruppierten Fakten herunter und überlassen die Aggregation sowie die Bereinigung von Links (z.B. URL-Rückgaben statt reinem Text bei der Schlaghand) der Bibliothek `pandas` in Python.

In [28]:
# Top 10 Home Run Hitter auswählen
top_df = df_csv.sort_values(by='HR', ascending=False).head(10)
player_names_subset = top_df['Name'].dropna().unique().tolist()

sparql_names_filter = ", ".join([f'"{name}"' for name in player_names_subset])
sparql = SPARQLWrapper("http://dbpedia.org/sparql")

# SPARQL Query mit dbp:bats (Schlaghand)
query = f"""
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX dbp: <http://dbpedia.org/property/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?name ?bdate ?bats
WHERE {{
  ?person a dbo:BaseballPlayer ;
          rdfs:label ?name .
          
  OPTIONAL {{ ?person dbo:birthDate ?bdate . }}
  OPTIONAL {{ ?person dbp:bats ?bats . }}
          
  FILTER(lang(?name) = "en")
  FILTER(str(?name) IN ({sparql_names_filter}))
}} 
"""

sparql.setQuery(query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

# Ergebnisse parsen
rdf_rows = []
for result in results["results"]["bindings"]:
    bats_raw = result["bats"]["value"] if "bats" in result else "Unknown"
    
    # Manchmal liefert DBpedia einen Link (http://...), wir wollen nur das letzte Wort (Right/Left)
    if "http" in bats_raw:
        bats_clean = bats_raw.split("/")[-1]
    else:
        bats_clean = bats_raw
        
    rdf_rows.append({
        "Name": result["name"]["value"],
        "BirthDate": result["bdate"]["value"] if "bdate" in result else None,
        "Bats": bats_clean
    })

# DataFrame erstellen, sortieren und Duplikate filtern
df_rdf = pd.DataFrame(rdf_rows)
df_rdf = df_rdf.sort_values(by=['BirthDate'], na_position='last').groupby('Name').first().reset_index()

print(f"ERFOLG: Biographische Daten (inkl. Schlaghand) für {len(df_rdf)} Spieler geladen.")
display(df_rdf)

ERFOLG: Biographische Daten (inkl. Schlaghand) für 9 Spieler geladen.


,Name,BirthDate,Bats
0,Aaron Judge,1992-04-26,Right
1,Cal Raleigh,1996-11-26,Switch
2,Jo Adell,1999-04-08,Right
3,Juan Soto,1998-10-25,Left
4,Junior Caminero,2003-07-05,Right
5,Kyle Schwarber,1993-03-05,Left
6,Pete Alonso,1994-12-07,Right
7,Shohei Ohtani,1994-07-05,Left
8,Taylor Ward,1993-12-14,Right


## 2. Relationale Datenhaltung: Erstellung der lokalen Datenbank

Um die Anforderungen an eine relationale DB-Applikation zu erfüllen, nutzen wir eine lokale, persistente SQLite-Datenbank (`baseball_2025.db`). 

Die aufbereiteten Leistungsdaten wandern in die Tabelle `player_stats`, die biographischen RDF-Daten in die Tabelle `player_biographies`. Der Parameter `if_exists='replace'` stellt die Idempotenz des Codes sicher – die Tabellen werden bei jedem Ausführen sauber neu generiert.

In [29]:
# Verbindung zu einer lokalen, persistenten SQLite-Datenbank herstellen
# Es entsteht physisch die Datei 'baseball_2025.db' auf Ihrer Festplatte
conn = sqlite3.connect('baseball_2025.db')

# Tabelle 1: Sportliche Statistiken (aus CSV) in DB schreiben
df_csv.to_sql('player_stats', conn, index=False, if_exists='replace')

# Tabelle 2: Biographische Daten (aus RDF/SPARQL) in DB schreiben
df_rdf.to_sql('player_biographies', conn, index=False, if_exists='replace')

print("Erfolg: Datei 'baseball_2025.db' wurde lokal erstellt und mit beiden Tabellen befüllt.")

Erfolg: Datei 'baseball_2025.db' wurde lokal erstellt und mit beiden Tabellen befüllt.


## 3. Datenintegration und Auswertung (Combine) via SQL

Da nun beide heterogenen Datenquellen strukturiert in unserem relationalen Schema liegen, kombinieren wir sie über einen SQL-`JOIN` mit dem gemeinsamen Schlüssel `Name`.

### Auswertung A: Die Top Home Run Hitter, ihr Alter und ihre Schlaghand
Wir filtern nach der maximalen sportlichen Leistung (Home Runs) und fügen das Alter der Spieler hinzu. Dieses wird dynamisch im SQL-Statement aus dem DBpedia-Geburtsdatum berechnet. Zudem zeigen wir, ob die besten Power-Hitter der Liga bevorzugt Links- oder Rechtshänder sind.

In [30]:
# SQL-Abfrage 1: Verknüpfung über JOIN, Altersberechnung und Anzeige der Schlaghand
query_hr = """
SELECT s.Name, b.Bats as Schlaghand, s.HR as HomeRuns, 
       (2025 - CAST(SUBSTR(b.BirthDate, 1, 4) AS INTEGER)) as Age
FROM player_stats s
JOIN player_biographies b ON s.Name = b.Name
ORDER BY s.HR DESC
"""

df_res_hr = pd.read_sql_query(query_hr, conn)
print("Auswertung A: Die Top Home Run Hitter, ihr Alter und ihre bevorzugte Schlaghand:")
display(df_res_hr)

Auswertung A: Die Top Home Run Hitter, ihr Alter und ihre bevorzugte Schlaghand:


,Name,Schlaghand,HomeRuns,Age
0,Cal Raleigh,Switch,60,29
1,Kyle Schwarber,Left,56,32
2,Shohei Ohtani,Left,55,31
3,Aaron Judge,Right,53,33
4,Junior Caminero,Right,45,22
5,Juan Soto,Left,43,27
6,Pete Alonso,Right,38,31
7,Jo Adell,Right,37,26
8,Taylor Ward,Right,36,32


### Auswertung B: Schlagdurchschnitt (AVG) im Verhältnis zum Alter
Da die Daten nun persistiert sind, können wir jederzeit neue Analysen fahren, ohne den DBpedia-Server erneut anfragen zu müssen. Hier untersuchen wir die sportliche Effizienz (Batting Average) der Spieler und ordnen diese ebenfalls dem Alter zu. Am Ende wird die Datenbankverbindung sauber geschlossen.

In [31]:
# SQL-Abfrage 2: Gleiche Logik, aber wir schauen auf die Effizienz (Batting Average)
query_avg = """
SELECT s.Name, s.AVG as BattingAverage, 
       (2025 - CAST(SUBSTR(b.BirthDate, 1, 4) AS INTEGER)) as Age
FROM player_stats s
JOIN player_biographies b ON s.Name = b.Name
ORDER BY s.AVG DESC
"""

df_res_avg = pd.read_sql_query(query_avg, conn)
print("Auswertung B: Sportliche Effizienz (AVG) im Verhältnis zum Alter:")
display(df_res_avg)

# Datenbankverbindung am Ende sauber schließen
conn.close()


Auswertung B: Sportliche Effizienz (AVG) im Verhältnis zum Alter:


,Name,BattingAverage,Age
0,Aaron Judge,0.331,33
1,Shohei Ohtani,0.282,31
2,Pete Alonso,0.272,31
3,Junior Caminero,0.264,22
4,Juan Soto,0.263,27
5,Cal Raleigh,0.247,29
6,Kyle Schwarber,0.240,32
7,Jo Adell,0.236,26
8,Taylor Ward,0.228,32
